# Day 059 — Exercise 3: Worker API (FastAPI + JobQueue)

Wire `JobQueue` from Exercise 2 into a FastAPI app. The API accepts jobs via `POST /jobs`, returns a `job_id` immediately (202 Accepted), and lets clients poll `GET /jobs/{job_id}` until the status is `done` or `error`.

This is the standard REST pattern for long-running tasks: accept immediately, process asynchronously, poll for completion.

In [ ]:
# --- Provided: JobQueue (from Exercise 2) ---
import threading
import uuid
from typing import Any, Callable

class JobQueue:
    def __init__(self):
        self._jobs: dict = {}
        self._lock = threading.Lock()

    def submit(self, fn: Callable, *args) -> str:
        job_id = uuid.uuid4().hex[:8]
        with self._lock:
            self._jobs[job_id] = {"status": "pending"}

        def worker():
            with self._lock:
                self._jobs[job_id]["status"] = "running"
            try:
                value = fn(*args)
                with self._lock:
                    self._jobs[job_id] = {"status": "done", "result": value}
            except Exception as exc:
                with self._lock:
                    self._jobs[job_id] = {"status": "error", "error": str(exc)}

        threading.Thread(target=worker, daemon=True).start()
        return job_id

    def status(self, job_id: str) -> str:
        with self._lock:
            return self._jobs.get(job_id, {}).get("status", "not_found")

    def result(self, job_id: str) -> Any:
        with self._lock:
            job = self._jobs.get(job_id, {})
            return job.get("result") if job.get("status") == "done" else None


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import time


## Task

Implement `build_worker_api(process_fn=None)` — return a FastAPI app with:

```
POST /jobs   {"payload": "..."}  → 202  {"job_id": "...", "status": "pending"}
GET  /jobs/{job_id}            → 200  {"status": ..., "result": ...}
                               → 404  if unknown
```

Use `JobQueue` (already defined above). Pass `process_fn` (or `str.upper`) to `queue.submit()` as the worker function.

## Your Implementation

In [ ]:
def build_worker_api(process_fn=None) -> FastAPI:
    """FastAPI app backed by a JobQueue.

    Endpoints:
      POST /jobs  {"payload": "..."}  → 202  {"job_id": "...", "status": "pending"}
      GET /jobs/{job_id}             → 200  {"status": ..., "result": ...}
                                     → 404 if unknown

    process_fn: optional callable(payload: str) -> str for testing.
                If None, uppercase the payload (trivial default).
    """
    # TODO: create JobQueue, build app with POST /jobs and GET /jobs/{job_id}
    raise NotImplementedError


In [ ]:
def build_worker_api(process_fn=None) -> FastAPI:
    app   = FastAPI()
    queue = JobQueue()

    class _JobReq(BaseModel):
        payload: str = Field(min_length=1)

    @app.post("/jobs", status_code=202)
    def submit(req: _JobReq):
        fn     = process_fn if process_fn is not None else str.upper
        job_id = queue.submit(fn, req.payload)
        return {"job_id": job_id, "status": "pending"}

    @app.get("/jobs/{job_id}")
    def get_job(job_id: str):
        s = queue.status(job_id)
        if s == "not_found":
            raise HTTPException(404, "Job not found")
        result = queue.result(job_id)
        return {"status": s, "result": result}

    return app


## Automated checks

In [ ]:
score, total = 0, 5

def _wait_job(client, job_id, timeout=2.0):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        r = client.get(f"/jobs/{job_id}")
        if r.json()["status"] not in ("pending", "running"):
            return r.json()
        time.sleep(0.01)
    return client.get(f"/jobs/{job_id}").json()

try:
    app    = build_worker_api(process_fn=str.upper)
    client = TestClient(app, raise_server_exceptions=False)

    r = client.post("/jobs", json={"payload": "hello"})
    assert r.status_code == 202, f"Expected 202, got {r.status_code}"
    score += 1; print("\u2705 POST /jobs returns 202")

    body = r.json()
    assert "job_id" in body, f"Expected job_id in {body}"
    score += 1; print("\u2705 response contains job_id")

    job = _wait_job(client, body["job_id"])
    assert job["status"] == "done", f"Expected done, got {job}"
    assert job["result"] == "HELLO", f"Expected HELLO, got {job['result']}"
    score += 1; print("\u2705 job completes with correct result")

    r2 = client.get("/jobs/no_such_id")
    assert r2.status_code == 404, f"Expected 404, got {r2.status_code}"
    score += 1; print("\u2705 unknown job_id → 404")

    r3 = client.post("/jobs", json={"payload": ""})
    assert r3.status_code == 422, f"Expected 422 for empty payload, got {r3.status_code}"
    score += 1; print("\u2705 empty payload → 422")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_worker_api(process_fn=None) -> FastAPI:
    app   = FastAPI()
    queue = JobQueue()

    class _JobReq(BaseModel):
        payload: str = Field(min_length=1)

    @app.post("/jobs", status_code=202)
    def submit(req: _JobReq):
        fn     = process_fn if process_fn is not None else str.upper
        job_id = queue.submit(fn, req.payload)
        return {"job_id": job_id, "status": "pending"}

    @app.get("/jobs/{job_id}")
    def get_job(job_id: str):
        s = queue.status(job_id)
        if s == "not_found":
            raise HTTPException(404, "Job not found")
        result = queue.result(job_id)
        return {"status": s, "result": result}

    return app
```

**Why 202 Accepted?** HTTP 200 means *done*. 202 means *accepted for processing* — exactly right when the work hasn't finished yet. Clients must poll `GET /jobs/{id}` to discover completion, which is why the response includes `job_id`.

</details>